In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import h5py, os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from msi.utils import preprocessing
from msfm.utils import prior, parameters, files, logger

In [13]:
conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v8/linear_bias.yaml")
base_dir = "/pscratch/sd/a/athomsen/run_files/v8"

# delta loss #####################################################################################################

# # lensing
# model_dir = "lensing/delta/2024-04-26_05-50-54_resnet_vanilla"
# params = ["Om", "s8", "w0", "Aia", "n_Aia"]
# n_steps = 100_000

# # clustering
# model_dir = "clustering/delta/2024-04-26_05-50-43_resnet_vanilla"
# params = ["Om", "s8", "w0", "bg", "n_bg"]
# n_steps = 80_000

# combined
model_dir = "combined/delta/2024-04-26_17-18-07_resnet_vanilla"
params = ["Om", "s8", "w0", "Aia", "n_Aia", "bg", "n_bg"]
n_steps = 50_000

In [14]:
# dataset
fidu_preds, grid_preds, grid_cosmos, file_dict = preprocessing.get_reshaped_network_preds(
    base_dir, model_dir, n_steps, n_params=len(params)
)

# output directory and file names
out_dir = os.path.join(base_dir, model_dir)
label = f"{n_steps}_steps"

24-04-29 04:53:07 input_output INF   Array shapes: 
24-04-29 04:53:07 input_output INF   fiducial/vali/pred = (4000, 7) 
24-04-29 04:53:07 input_output INF   fiducial/vali/i_example = (4000,) 
24-04-29 04:53:07 input_output INF   fiducial/vali/i_noise = (4000,) 
24-04-29 04:53:07 input_output INF   grid/pred          = (2500, 48, 7) 
24-04-29 04:53:07 input_output INF   grid/cosmo         = (2500, 7) 
24-04-29 04:53:07 input_output INF   grid/i_example     = (2500, 48) 
24-04-29 04:53:07 input_output INF   grid/i_noise       = (2500, 48) 
24-04-29 04:53:07 input_output INF   grid/i_sobol       = (2500,) 


24-04-29 04:53:07 preprocessin INF   Shapes after concatenation and selection: 
24-04-29 04:53:07 preprocessin INF   fidu_preds  = (4000, 7) 
24-04-29 04:53:07 preprocessin INF   grid_preds  = (120000, 7) 
24-04-29 04:53:07 preprocessin INF   grid_cosmos = (120000, 7) 


In [38]:
grid_cosmo = file_dict["grid/cosmo"]
grid_sobol = file_dict["grid/i_sobol"]

fiducial = parameters.get_fiducials(params)

standard_scaler = StandardScaler()

cosmo = standard_scaler.fit_transform(grid_cosmo)
# cosmo = grid_cosmos
# cosmo_diff = np.square(cosmo - fiducial)
cosmo_diff = np.abs(cosmo - fiducial)
cosmo_diff = cosmo_diff[:2]
cosmo_diff = np.mean(cosmo_diff, axis=1)

i_closest_grid_cosmo = np.argmin(cosmo_diff)

print(fiducial)
print(i_closest_grid_cosmo, grid_cosmo[i_closest_grid_cosmo], grid_sobol[i_closest_grid_cosmo])

[ 0.26  0.84 -1.    0.5   1.5   1.5   0.5 ]
0 [ 0.3     0.9    -1.1665  0.      1.      1.9     0.    ] 1


In [30]:
# fiducial = parameters.get_fiducials(params)

# standard_scaler = StandardScaler()

# # cosmo = standard_scaler.fit_transform(grid_cosmos)
# cosmo = grid_cosmos
# cosmo_diff = np.square(cosmo - fiducial)
# cosmo_diff = np.mean(cosmo_diff, axis=1)

# i_closest_grid_cosmo = np.argmin(cosmo_diff)

# print(fiducial)
# print(i_closest_grid_cosmo, grid_cosmos[i_closest_grid_cosmo])
# # print(grid_cosmos[i_closest_grid_cosmo, file_dict["grid/i_sobol"])

[ 0.26  0.84 -1.    0.5   1.5   1.5   0.5 ]
63504 [ 0.22785644  1.1011108  -0.8491555   0.52697754  1.3448486   1.2185425
  0.6265869 ]
